# Laras Karya: Pipeline Rekomendasi Lowongan Hijau
### Hands-on Google Colab: Rule-based Scoring + Penjelasan AI (dataset `green-jobs.csv`)

**Tujuan pembelajaran**
- Memuat dataset lowongan hijau (`CSV`).
- Memahami struktur dan membersihkan data.
- Memfilter lowongan dengan aturan keras (hard filter).
- Memberi score kesesuaian profil dengan **tujuh komponen berbobot**.
- Mengurutkan lowongan dan menjelaskan alasannya (rule-based).
- Memakai AI hanya untuk **menafsirkan** hasil aturan dan mengusulkan skill dari cerita pengalaman.
- Mengevaluasi pipeline dengan uji otomatis sederhana.

**Catatan penting**
- Notebook ini **tidak terhubung ke Supabase**. Semua data berasal dari file `green-jobs.csv`.
- Score adalah ukuran *kesesuaian* profil dengan lowongan, **bukan probabilitas diterima kerja**.
- Weight adalah nilai awal yang perlu dikalibrasi dengan data nyata.
- Penjelasan AI dan ekstraksi skill di notebook ini masih **simulasi** (template dan kata kunci, bukan LLM sungguhan), sehingga bisa dijalankan tanpa API key.
- Jangan gunakan atribut sensitif (misalnya ras, agama, gender, disabilitas, atau usia) untuk memberi peringkat kandidat.

**Prinsip desain: terkontrol dulu, baru AI.**

| Lapisan | Tugas | Mengapa begitu |
|---|---|---|
| **1. Rule-based** | Hard filter, weighted scoring 7 komponen, dan ranking | Deterministik, bisa diaudit, dan tidak bisa "berhalusinasi" lowongan |
| **2. AI (LLM)** | (a) hilir: menulis penjelasan personal untuk Top-K hasil aturan; (b) hulu: mengusulkan skill dari cerita pengalaman, baru dipakai setelah dikonfirmasi pengguna | AI dipakai di tempat ia kuat (bahasa), bukan untuk memutuskan siapa layak |

**10 tahapan:** 1 Setup data · 2 Kolom & konfigurasi · 3 Cleaning · 4 Profil kandidat · 5 Ekstraksi skill · 6 Filter & score · 7 Rank & explain · 8 Eksperimen weight · 9 Lapisan AI · 10 Uji & kesimpulan

**Alur pipeline**
```
Cerita pengalaman ─► [EKSTRAKSI SKILL (AI)] ─► pengguna konfirmasi ─► skill profil
                                                                          │
green-jobs.csv ─► [CLEANING] ─► [FILTER] ─► [SCORE 7 komponen] ─► [RANK Top-N]
                                                                          │
                                  [AI: penjelasan + next step, hanya Top-K] ─► [VALIDASI guardrail] ─► CSV hasil
```

## 1. Setup dan upload data
Install dependency, upload `green-jobs.csv`, lalu periksa kolom dan kualitas datanya.

In [1]:
!pip -q install pandas numpy

In [2]:
import io
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

try:
    from google.colab import files

    uploaded = files.upload()

    if not uploaded:
        raise ValueError("Belum ada file yang di-upload.")

    csv_name = next(iter(uploaded))
    df_jobs = pd.read_csv(io.BytesIO(uploaded[csv_name]))
except ImportError:
    # Bukan di Colab: baca langsung dari folder kerja
    csv_name = "green-jobs.csv"
    df_jobs = pd.read_csv(csv_name)

print(f"File: {csv_name}")
print(f"Ukuran dataset: {df_jobs.shape[0]:,} baris × {df_jobs.shape[1]} kolom")
display(df_jobs.head())

File: green-jobs.csv
Ukuran dataset: 50 baris × 10 kolom


,id,job_title,skills,company_name,sector,benefits,province,experience_level,min_education,work_arrangement
0,95465065,Environment Officer,"[Environmental regulations & compliance, Waste...",Anzawara Satria,environmental-officer,"Allowances, Medical Insurance, Medical Reimbui...",Tanah Bumbu,NaN,S1,On-site
1,95025434,Supervisor Environment,"[Environmental management, Environmental regul...",Priamanaya Energy,supervisor,NaN,Lahat,Intermediate,S1,On-site
2,95355234,"HSE (Health, Safety and Environment) MANAGER","[Environmental regulations & compliance, Waste...",CICOR Panatec,health-safety-and-environment-manager,"Medical, Bonus",Batam,Advanced,S1,On-site
3,95527538,Oil Refinery,"[Environmental management, Environmental regul...",Human Protalent Indonesia,NaN,"Medical, BPJS Kesehatan dan Ketenagakerjaan",NaN,NaN,S1,On-site
4,95437400,HSEQ Specialist,"[HSE / HSEQ / QHSE management, Safety training...",Jotun,"health,-safety,-environment-and-quality-specia...","Medical, Miscellaneous allowance, Education su...",Bekasi,NaN,NaN,On-site


In [3]:
print("Daftar kolom:")
for col in df_jobs.columns:
    print("-", col)

display(df_jobs.isna().sum().sort_values(ascending=False).to_frame("missing_values"))

# Nilai pada kolom kategori (kolom dengan maksimal 10 nilai unik)
for col in df_jobs.columns:
    if df_jobs[col].nunique(dropna=True) <= 10:
        display(df_jobs[col].value_counts(dropna=False).to_frame(col))

print("Contoh isi kolom skills:")
print(df_jobs["skills"].iloc[0] if "skills" in df_jobs.columns else "-")

Daftar kolom:
- id
- job_title
- skills
- company_name
- sector
- benefits
- province
- experience_level
- min_education
- work_arrangement


,missing_values
experience_level,34
benefits,25
province,16
company_name,11
min_education,9
sector,5
job_title,0
id,0
skills,0
work_arrangement,0


,experience_level
experience_level,
NaN,34
Intermediate,9
Advanced,7


,min_education
min_education,
S1,32
NaN,9
SMA,6
D3,3


,work_arrangement
work_arrangement,
On-site,49
Hybrid,1


Contoh isi kolom skills:
[Environmental regulations & compliance, Waste management (B3/non-B3), AMDAL / UKL-UPL / RKL-RPL, PROPER, Safety training / induction / toolbox meeting, SOP / procedures / JSA / permit to work, GIS (ArcGIS / QGIS / Global Mapper), Laboratory operations & QA/QC]


## 2. Deteksi kolom dan konfigurasi
Kolom dideteksi otomatis. **Weight tidak diubah** dari pipeline awal (total 1,0); nilai `NEUTRAL = 50` dipakai bila data missing.

In [4]:
def find_column(columns, candidates):
    # Pencocokan nama kolom tanpa membedakan huruf besar/kecil
    normalized = {str(c).strip().lower(): c for c in columns}
    for candidate in candidates:
        if candidate.lower() in normalized:
            return normalized[candidate.lower()]
    # Coba pencocokan sebagian nama
    for c in columns:
        cl = str(c).strip().lower()
        if any(candidate.lower() in cl for candidate in candidates):
            return c
    return None

id_col = find_column(df_jobs.columns, ["id", "job_id"])
title_col = find_column(df_jobs.columns, ["job_title", "title", "position", "role"])
company_col = find_column(df_jobs.columns, ["company_name", "company", "employer"])
skills_col = find_column(df_jobs.columns, ["skills", "requisite_skill", "required_skills"])
sector_col = find_column(df_jobs.columns, ["sector", "industry"])
benefits_col = find_column(df_jobs.columns, ["benefits", "benefit"])
location_col = find_column(df_jobs.columns, ["province", "location", "workplace", "city"])
experience_col = find_column(df_jobs.columns, ["experience_level", "experience", "seniority"])
education_col = find_column(df_jobs.columns, ["min_education", "education", "minimum_education"])
mode_col = find_column(df_jobs.columns, ["work_arrangement", "working_mode", "work_mode"])

# Opsional (tidak ada di green-jobs.csv)
status_col = find_column(df_jobs.columns, ["status"])
green_col = find_column(df_jobs.columns, ["is_green"])
deadline_col = find_column(df_jobs.columns, ["deadline", "closing_date"])

print("ID lowongan:", id_col)
print("Judul pekerjaan:", title_col)
print("Perusahaan:", company_col)
print("Skill:", skills_col)
print("Sektor:", sector_col)
print("Benefit:", benefits_col)
print("Lokasi:", location_col)
print("Level pengalaman:", experience_col)
print("Pendidikan minimum:", education_col)
print("Pola kerja:", mode_col)
print("Status / is_green / deadline (opsional):", status_col, "/", green_col, "/", deadline_col)

# Jika deteksi tidak tepat, isi nama kolom secara manual, misalnya:
# title_col = "job_title"
# skills_col = "skills"
# location_col = "province"

if title_col is None or skills_col is None:
    raise ValueError("Kolom judul pekerjaan atau skill tidak terdeteksi. Atur title_col / skills_col secara manual.")

ID lowongan: id
Judul pekerjaan: job_title
Perusahaan: company_name
Skill: skills
Sektor: sector
Benefit: benefits
Lokasi: province
Level pengalaman: experience_level
Pendidikan minimum: min_education
Pola kerja: work_arrangement
Status / is_green / deadline (opsional): None / None / None


In [5]:
import re
import json
import hashlib
from datetime import date, datetime

# Weight tiap komponen score. Total harus 1.0. Ubah di sini saja.
WEIGHTS = {
    "skill": 0.25,       # skill user vs skill yang diminta lowongan
    "sector": 0.20,      # minat sektor hijau
    "work_style": 0.10,  # onsite / remote / hybrid
    "location": 0.10,    # provinsi / remote / bersedia relokasi
    "experience": 0.15,  # level pengalaman
    "education": 0.10,   # jenjang pendidikan minimum
    "benefits": 0.10,    # jam fleksibel, laktasi, return-to-work, dst.
}
assert abs(sum(WEIGHTS.values()) - 1.0) < 1e-9, "Total WEIGHTS harus 1.0"

TOP_N = 20
NEUTRAL = 50.0           # nilai pengganti bila data missing (bukan berarti tidak cocok)
EXCLUDE_APPLIED = False  # True = lowongan yang sudah dilamar disembunyikan dari hasil

# Urutan jenjang (angka lebih besar = lebih tinggi)
EDUCATION_RANK = {"sma": 1, "d3": 2, "s1": 3, "s2": 4, "s3": 5}
EXPERIENCE_RANK = {"entry": 0, "junior": 1, "mid": 2, "senior": 3}

# Nilai mentah dataset -> enum standar di atas
EDUCATION_ALIASES = {"sma": "sma", "smk": "sma", "sma/smk": "sma", "d3": "d3", "s1": "s1", "s2": "s2", "s3": "s3"}
EXPERIENCE_ALIASES = {"entry": "entry", "entry_level": "entry", "junior": "junior",
                      "intermediate": "mid", "mid": "mid", "advanced": "senior", "senior": "senior"}
ARRANGEMENT_ALIASES = {"onsite": "onsite", "wfo": "onsite", "remote": "remote", "wfh": "remote", "hybrid": "hybrid"}

# Kebutuhan user (preferences) -> benefit yang dicari di lowongan.
NEED_TO_BENEFIT = {
    "needs_flexible_hours": "jam_fleksibel",
    "has_career_break": "return_to_work",
    "needs_childcare": "penitipan_anak",
    "needs_lactation_room": "fasilitas_laktasi",
}
# Keywords agar 'jam kerja fleksibel', 'jam-fleksibel', dan 'Fleksibilitas kerja' dianggap sama.
BENEFIT_KEYWORDS = {
    "jam_fleksibel": ["fleksib", "flexib"],
    "return_to_work": ["return"],
    "penitipan_anak": ["penitipan", "daycare"],
    "fasilitas_laktasi": ["laktasi"],
}
BENEFIT_LABEL = {
    "jam_fleksibel": "jam kerja fleksibel",
    "return_to_work": "program return-to-work",
    "penitipan_anak": "penitipan anak",
    "fasilitas_laktasi": "fasilitas laktasi",
}

display(pd.DataFrame({"weight": WEIGHTS}).assign(persen=lambda d: (d["weight"] * 100).round(0).astype(int).astype(str) + "%"))
print("Konfigurasi siap. Total weight =", round(sum(WEIGHTS.values()), 2))

,weight,persen
skill,0.25,25%
sector,0.20,20%
work_style,0.10,10%
location,0.10,10%
experience,0.15,15%
education,0.10,10%
benefits,0.10,10%


Konfigurasi siap. Total weight = 1.0


## 3. Normalisasi dan cleaning
Teks diseragamkan dan skill dipecah dengan benar (nama skill memuat koma). `[Perlu review manual]` = skill missing. Hasilnya tabel `df_jobs` dengan kolom bantu berawalan `_`.

In [6]:
def norm(text):
    # 'Pelaporan ESG' / 'pelaporan-esg' -> 'pelaporan_esg'
    return "_".join(str(text).strip().lower().replace("-", " ").split())


def norm_set(values):
    return {norm(v) for v in (values or []) if v and str(v).strip()}


def split_items(text, split_on_capital=False):
    # '[A, B, C]' -> ['A', 'B', 'C']. '[Perlu review manual]' dan teks kosong -> []
    text = str(text).strip()
    if text.startswith("[") and text.endswith("]"):
        text = text[1:-1]
    if not text or "perlu review manual" in text.lower():
        return []
    pattern = r",\s*(?=[A-Z0-9])" if split_on_capital else r",\s*"
    return [t.strip() for t in re.split(pattern, text) if t.strip()]


def norm_benefits(values):
    # Free-text benefit -> kode benefit standar (lihat BENEFIT_KEYWORDS)
    texts = [norm(v) for v in (values or []) if v]
    return {code for code, keys in BENEFIT_KEYWORDS.items() if any(k in t for t in texts for k in keys)}


def norm_arrangement(text):
    # 'On-site' -> 'onsite'. Nilai tidak dikenal -> ''
    key = re.sub(r"[^a-z]", "", str(text).lower())
    return ARRANGEMENT_ALIASES.get(key, "")


def norm_experience(text):
    # 'Intermediate' -> 'mid', 'Advanced' -> 'senior'. Nilai tidak dikenal -> ''
    return EXPERIENCE_ALIASES.get(norm(text), "")


def norm_education(text):
    # 'S1' -> 's1'. Nilai tidak dikenal -> ''
    return EDUCATION_ALIASES.get(str(text).strip().lower(), "")


def norm_location(text):
    # 'Bekasi Regency' dan 'Bekasi' dianggap sama; kata 'Regency/City/Kabupaten/Kota' dibuang
    words = [w for w in str(text).lower().split() if w not in {"regency", "city", "kabupaten", "kota"}]
    return "_".join(words)


def parse_date(value):
    # Terima 'YYYY-MM-DD', timestamp ISO, date, atau datetime. Kosong -> None
    if value is None or str(value).strip() in ("", "nan", "NaT", "None"):
        return None
    if isinstance(value, datetime):
        return value.date()
    if isinstance(value, date):
        return value
    return date.fromisoformat(str(value)[:10])


print(split_items("[Quality, safety & environmental auditing, Mining operations, Communication]", split_on_capital=True))
print(split_items("[Perlu review manual]", split_on_capital=True))
print(norm_arrangement("On-site"), norm_experience("Intermediate"), norm_education("S1"), norm_location("Bekasi Regency"))

['Quality, safety & environmental auditing', 'Mining operations', 'Communication']
[]
onsite mid s1 bekasi


In [7]:
def safe_text(row, col):
    if col is None:
        return ""
    value = row.get(col, "")
    if pd.isna(value):
        return ""
    return str(value).strip()


def is_green_value(value):
    # Kosong dianggap hijau; hanya False / 0 / no yang dianggap bukan
    return str(value).strip().lower() not in ("false", "0", "no", "n")


df_jobs = df_jobs.copy()

df_jobs["_id"] = df_jobs.apply(lambda r: safe_text(r, id_col) or str(r.name), axis=1)
df_jobs["_title"] = df_jobs.apply(lambda r: safe_text(r, title_col), axis=1)
df_jobs["_company"] = df_jobs.apply(lambda r: safe_text(r, company_col), axis=1)
df_jobs["_sector"] = df_jobs.apply(lambda r: safe_text(r, sector_col), axis=1)
df_jobs["_location"] = df_jobs.apply(lambda r: safe_text(r, location_col), axis=1)
df_jobs["_mode"] = df_jobs.apply(lambda r: norm_arrangement(safe_text(r, mode_col)), axis=1)
df_jobs["_experience"] = df_jobs.apply(lambda r: norm_experience(safe_text(r, experience_col)), axis=1)
df_jobs["_education"] = df_jobs.apply(lambda r: norm_education(safe_text(r, education_col)), axis=1)

# Skill: kode -> label asli
df_jobs["_skill_labels"] = df_jobs.apply(
    lambda r: {norm(s): s for s in split_items(safe_text(r, skills_col), split_on_capital=True)}, axis=1)
df_jobs["_skills"] = df_jobs["_skill_labels"].apply(set)

# Benefit: free text -> kode benefit standar. Teks asli disimpan untuk ditampilkan
df_jobs["_benefits_raw"] = df_jobs.apply(lambda r: safe_text(r, benefits_col), axis=1)
df_jobs["_benefits"] = df_jobs["_benefits_raw"].apply(lambda t: norm_benefits(split_items(t)))

# Kolom opsional: bila tidak ada, semua lowongan dianggap published, hijau, tanpa deadline
df_jobs["_status"] = df_jobs.apply(lambda r: safe_text(r, status_col).lower() or "published", axis=1)
df_jobs["_is_green"] = df_jobs.apply(lambda r: is_green_value(safe_text(r, green_col) or "true"), axis=1)
df_jobs["_deadline"] = df_jobs.apply(lambda r: safe_text(r, deadline_col)[:10], axis=1)

# Hapus baris tanpa judul dan ID ganda
n_before = len(df_jobs)
df_jobs = df_jobs[df_jobs["_title"].str.len() > 0]
df_jobs = df_jobs.drop_duplicates(subset=["_id"]).reset_index(drop=True)

# Nilai mentah yang tidak berhasil dipetakan ke enum standar
for raw_col, clean_col, label in [(experience_col, "_experience", "experience_level"),
                                  (education_col, "_education", "min_education"),
                                  (mode_col, "_mode", "work_arrangement")]:
    if raw_col is None:
        continue
    unknown = df_jobs[(df_jobs[raw_col].notna()) & (df_jobs[clean_col] == "")][raw_col].unique()
    if len(unknown):
        print(f"Perhatian: nilai {label} tidak dikenali dan dianggap missing:", list(unknown))

print(f"Lowongan setelah cleaning: {len(df_jobs):,} dari {n_before:,}")
print(f"Tanpa skill (perlu review manual / kosong): {(df_jobs['_skills'].apply(len) == 0).sum()}")
print(f"Tanpa lokasi: {(df_jobs['_location'] == '').sum()} | tanpa level pengalaman: {(df_jobs['_experience'] == '').sum()} | tanpa pendidikan minimum: {(df_jobs['_education'] == '').sum()}")
display(df_jobs[["_id", "_title", "_company", "_sector", "_location", "_mode", "_experience", "_education"]].head(10))

Lowongan setelah cleaning: 50 dari 50
Tanpa skill (perlu review manual / kosong): 2
Tanpa lokasi: 16 | tanpa level pengalaman: 34 | tanpa pendidikan minimum: 9


,_id,_title,_company,_sector,_location,_mode,_experience,_education
0,95465065,Environment Officer,Anzawara Satria,environmental-officer,Tanah Bumbu,onsite,,s1
1,95025434,Supervisor Environment,Priamanaya Energy,supervisor,Lahat,onsite,mid,s1
2,95355234,"HSE (Health, Safety and Environment) MANAGER",CICOR Panatec,health-safety-and-environment-manager,Batam,onsite,senior,s1
3,95527538,Oil Refinery,Human Protalent Indonesia,,,onsite,,s1
4,95437400,HSEQ Specialist,Jotun,"health,-safety,-environment-and-quality-specia...",Bekasi,onsite,,
5,95464915,Tyre Position (Supervisor & Superintendent Level),Macmahon,superintendent,,onsite,,sma
6,95463483,Auto Electrician (Supervisor & Superintendent ...,Macmahon,electrical-supervisor,,onsite,,sma
7,95284787,QC Supervisor,,quality-control-supervisor,Bekasi Regency,onsite,mid,s1
8,95283657,Environment Engineer,Petrosea,environmental-engineer,,onsite,mid,s1
9,95573223,Project Compliance Officer,Merdeka Copper Gold,compliance-officer,South Jakarta,onsite,mid,


## 4. Profil kandidat
Profil berbentuk dictionary (skill, minat sektor, pola kerja, lokasi, pengalaman, pendidikan). Gunakan nama skill dan sektor persis seperti di dataset.

In [8]:
def build_candidate(profile, educations=(), applied_job_ids=()):
    # Bangun profil kandidat yang sudah dinormalisasi
    ranks = [EDUCATION_RANK[norm_education(e)] for e in educations if norm_education(e) in EDUCATION_RANK]
    top = max(ranks, default=None)
    return {
        "id": str(profile["id"]),
        "skills": norm_set(profile.get("skills")),
        "interest_sectors": norm_set(profile.get("interest_sectors")),
        "work_arrangement_pref": norm_arrangement(profile.get("work_arrangement_pref") or "") or None,
        "province": profile.get("province"),
        "open_to_relocate": bool(profile.get("open_to_relocate")),
        "career_path": profile.get("career_path"),
        "experience_level": norm_experience(profile.get("experience_level") or "") or None,
        "preferences": profile.get("preferences") or {},   # sensitif: tidak boleh dikirim ke AI
        "max_education": next((k for k, v in EDUCATION_RANK.items() if v == top), None),
        "applied_job_ids": {str(j) for j in applied_job_ids},
    }


# Kosakata skill dan sektor yang ada di dataset
SKILL_VOCAB = {}
for labels in df_jobs["_skill_labels"]:
    SKILL_VOCAB.update(labels)
SKILL_VOCAB = dict(sorted(SKILL_VOCAB.items()))
SECTOR_OPTIONS = sorted(s for s in df_jobs["_sector"].unique() if s)
print(f"Skill unik di dataset: {len(SKILL_VOCAB)} | sektor/peran unik: {len(SECTOR_OPTIONS)}")


def check_known(candidate):
    # Skill atau sektor yang tidak ada di dataset tidak akan pernah cocok: beri peringatan
    unknown_skills = candidate["skills"] - set(SKILL_VOCAB)
    unknown_sectors = candidate["interest_sectors"] - {norm(s) for s in SECTOR_OPTIONS}
    if unknown_skills:
        print("Perhatian, skill tidak ada di dataset:", sorted(unknown_skills))
    if unknown_sectors:
        print("Perhatian, sektor tidak ada di dataset:", sorted(unknown_sectors))


# Profil demo: kembali bekerja setelah jeda karier, domisili Jakarta Selatan.
candidate = build_candidate(
    {
        "id": "demo-user",
        "skills": ["Microsoft Office / Excel", "Communication", "Reporting & presentation", "Problem solving"],
        "interest_sectors": ["environmental-officer", "environmental-engineer", "health-safety-and-environment-officer"],
        "work_arrangement_pref": "on-site",
        "province": "South Jakarta",
        "open_to_relocate": False,
        "career_path": "returning_to_work",
        "experience_level": "mid",
        "preferences": {"needs_flexible_hours": False},
    },
    educations=["s1"],
    applied_job_ids=["95283657"],   # Environment Engineer (Petrosea): sudah dilamar
)
check_known(candidate)

print("Profil demo  :", candidate["career_path"], "|", candidate["work_arrangement_pref"], "|", candidate["province"])
print("Skill        :", sorted(candidate["skills"]))
print("Sektor minat :", sorted(candidate["interest_sectors"]))
print("Kebutuhan    :", sorted(k for k, v in candidate["preferences"].items() if v is True))

Skill unik di dataset: 70 | sektor/peran unik: 40
Profil demo  : returning_to_work | onsite | South Jakarta
Skill        : ['communication', 'microsoft_office_/_excel', 'problem_solving', 'reporting_&_presentation']
Sektor minat : ['environmental_engineer', 'environmental_officer', 'health_safety_and_environment_officer']
Kebutuhan    : []


## 5. Ekstraksi skill dari cerita (AI di hulu)
**AI mengusulkan, aturan memutuskan.** Skill baru masuk profil hanya setelah dikonfirmasi pengguna, harus dari kosakata dataset, dan wajib disertai kutipan yang benar-benar ada di cerita. Tanpa persetujuan, LLM tidak dipanggil.

> `stub_skill_llm` adalah pencocokan kata kunci, bukan LLM sungguhan. Lima output salah diuji dan semuanya ditolak.

In [9]:
# Kata yang tidak boleh muncul pada kutipan maupun penjelasan AI (dipakai juga di tahap 9).
SENSITIVE_TERMS = ["hamil", "menyusui", "suami", "janda", "anak anda"]

# Cerita demo (experience_story). Data buatan sendiri.
DEMO_STORY = (
    "Selama tiga tahun saya bekerja sebagai petugas lingkungan di sebuah perusahaan tambang. "
    "Saya mengelola limbah B3 dan non-B3 di area operasional, menyusun laporan pemantauan lingkungan "
    "untuk instansi pemerintah, dan memastikan perusahaan memenuhi regulasi lingkungan. "
    "Saya juga memimpin sesi toolbox meeting keselamatan setiap pagi dan mengolah data di Excel. "
    "Setelah itu saya berhenti bekerja untuk mengurus keluarga dan kini ingin kembali bekerja "
    "di bidang lingkungan dan keselamatan kerja."
)
DEMO_CONSENT = True      # pada aplikasi nyata: pengguna sudah menyetujui pemrosesan oleh LLM

SKILL_SYSTEM_PROMPT = """Kamu membantu mengidentifikasi skill dari cerita pengalaman seseorang.
Aturan:
1. Pilih HANYA dari daftar VOCAB_JSON (gunakan field "code").
2. Untuk setiap skill sertakan "evidence": kutipan PERSIS (maks 15 kata) dari cerita yang mendukungnya. Tanpa kutipan = jangan sertakan skill itu.
3. Jangan menebak skill yang tidak didukung cerita, dan jangan mengutip informasi pribadi (keluarga, kesehatan, usia).
4. Balas HANYA dengan JSON: {"skills": [{"code": "...", "evidence": "..."}]}"""


def build_skill_prompt(story):
    vocab = [{"code": c, "label": l} for c, l in SKILL_VOCAB.items()]
    return SKILL_SYSTEM_PROMPT + "\n\nVOCAB_JSON:\n" + json.dumps(vocab, ensure_ascii=False) + "\n\nSTORY:\n" + story


# SIMULASI: pencocokan kata kunci (bukan LLM). Hanya untuk menguji alur dan guardrail tanpa API key.
# Kunci = label skill persis seperti di dataset.
STUB_PATTERNS = {
    "Waste management (B3/non-B3)": r"limbah B3(?: dan non-B3)?",
    "Environmental reporting & documentation": r"laporan pemantauan lingkungan",
    "Environmental monitoring": r"pemantauan lingkungan",
    "Environmental regulations & compliance": r"regulasi lingkungan",
    "Safety training / induction / toolbox meeting": r"toolbox meeting(?: \w+){0,1}",
    "Stakeholder / government relations": r"instansi pemerintah",
    "Leadership & team management": r"memimpin(?: \w+){0,2}",
    "Microsoft Office / Excel": r"\bexcel\b",
}
missing_labels = [l for l in STUB_PATTERNS if norm(l) not in SKILL_VOCAB]
assert not missing_labels, f"Label tidak ada di dataset: {missing_labels}"


def stub_skill_llm(prompt):
    story = prompt.split("STORY:\n", 1)[1]
    items = []
    for label, pattern in STUB_PATTERNS.items():
        m = re.search(pattern, story, flags=re.IGNORECASE)
        if m:
            items.append({"code": norm(label), "evidence": m.group(0)})
    return json.dumps({"skills": items}, ensure_ascii=False)


def validate_skill_output(raw, story, vocab, max_items=10):
    # Guardrail: JSON valid, kode ada di kosakata, kutipan ada di cerita (anti-halusinasi), tanpa kata sensitif
    try:
        data = json.loads(raw)
    except (json.JSONDecodeError, TypeError):
        return []
    items = data.get("skills", []) if isinstance(data, dict) else []
    story_l, seen, out = story.lower(), set(), []
    for it in items:
        if not isinstance(it, dict):
            continue
        code, ev = it.get("code"), it.get("evidence")
        if not isinstance(code, str) or code not in vocab or code in seen:
            continue
        if not isinstance(ev, str) or not ev.strip():
            continue
        ev = ev.strip()
        if len(ev.split()) > 15 or ev.lower() not in story_l:
            continue
        if any(t in ev.lower() for t in SENSITIVE_TERMS):
            continue
        seen.add(code)
        out.append({"code": code, "label": vocab[code], "evidence": ev})
    return out[:max_items]


def extract_skills(story, consent, llm=stub_skill_llm):
    # Cerita -> saran skill. Tanpa persetujuan atau cerita kosong, LLM tidak dipanggil. Error apa pun -> []
    if not consent or not story or not story.strip():
        return []
    try:
        raw = llm(build_skill_prompt(story))
    except Exception as e:
        print(f"[ekstraksi gagal -> tanpa saran] {type(e).__name__}: {e}")
        return []
    return validate_skill_output(raw, story, SKILL_VOCAB)


def apply_confirmed_skills(candidate, confirmed):
    # Hanya kode yang ada di kosakata DAN dikonfirmasi pengguna yang masuk ke profil
    return {**candidate, "skills": candidate["skills"] | (confirmed & set(SKILL_VOCAB))}


extracted = extract_skills(DEMO_STORY, consent=DEMO_CONSENT)
display(pd.DataFrame([
    {"kode": e["code"], "label": e["label"], "kutipan_dari_cerita": e["evidence"],
     "status": "sudah ada di profil" if e["code"] in candidate["skills"] else "BARU (menunggu konfirmasi)"}
    for e in extracted
]))

# Simulasi layar konfirmasi: pengguna menekan "Ya" pada semua saran BARU.
confirmed_codes = {e["code"] for e in extracted if e["code"] not in candidate["skills"]}
candidate_enriched = apply_confirmed_skills(candidate, confirmed_codes)
print("Dikonfirmasi pengguna :", len(confirmed_codes), "skill baru")
print("Jumlah skill sebelum  :", len(candidate["skills"]))
print("Jumlah skill sesudah  :", len(candidate_enriched["skills"]))
print("Disimpan              : hanya kode skill (cerita dan kutipan tidak disimpan)")

,kode,label,kutipan_dari_cerita,status
0,waste_management_(b3/non_b3),Waste management (B3/non-B3),limbah B3 dan non-B3,BARU (menunggu konfirmasi)
1,environmental_reporting_&_documentation,Environmental reporting & documentation,laporan pemantauan lingkungan,BARU (menunggu konfirmasi)
2,environmental_monitoring,Environmental monitoring,pemantauan lingkungan,BARU (menunggu konfirmasi)
3,environmental_regulations_&_compliance,Environmental regulations & compliance,regulasi lingkungan,BARU (menunggu konfirmasi)
4,safety_training_/_induction_/_toolbox_meeting,Safety training / induction / toolbox meeting,toolbox meeting keselamatan,BARU (menunggu konfirmasi)
5,stakeholder_/_government_relations,Stakeholder / government relations,instansi pemerintah,BARU (menunggu konfirmasi)
6,leadership_&_team_management,Leadership & team management,memimpin sesi toolbox,BARU (menunggu konfirmasi)
7,microsoft_office_/_excel,Microsoft Office / Excel,Excel,sudah ada di profil


Dikonfirmasi pengguna : 7 skill baru
Jumlah skill sebelum  : 4
Jumlah skill sesudah  : 11
Disimpan              : hanya kode skill (cerita dan kutipan tidak disimpan)


In [10]:
story_sensitive = "Saya menyusui sambil mengelola limbah B3 di area tambang."
code_waste = norm("Waste management (B3/non-B3)")

bad_extractions = {
    "kode di luar kosakata": (DEMO_STORY, json.dumps({"skills": [{"code": "kepemimpinan_galaksi", "evidence": "limbah B3"}]})),
    "kutipan karangan (tidak ada di cerita)": (DEMO_STORY, json.dumps({"skills": [{"code": norm("SQL / database"), "evidence": "saya mahir SQL"}]})),
    "tanpa kutipan": (DEMO_STORY, json.dumps({"skills": [{"code": norm("GIS (ArcGIS / QGIS / Global Mapper)")}]})),
    "kutipan memuat info pribadi": (story_sensitive, json.dumps({"skills": [{"code": code_waste, "evidence": "menyusui sambil mengelola limbah B3"}]})),
    "bukan JSON": (DEMO_STORY, "Skill Anda antara lain GIS dan SQL."),
}
for name, (story, raw) in bad_extractions.items():
    assert validate_skill_output(raw, story, SKILL_VOCAB) == [], f"Guardrail gagal: {name}"
    print(f"DITOLAK ✔  {name}")


def must_not_call(prompt):
    raise AssertionError("LLM tidak boleh dipanggil tanpa persetujuan")

assert extract_skills(DEMO_STORY, consent=False, llm=must_not_call) == []
print("TANPA PERSETUJUAN ✔  LLM tidak dipanggil")


def broken_llm(prompt):
    raise TimeoutError("API tidak merespons")

assert extract_skills(DEMO_STORY, consent=True, llm=broken_llm) == []
print("FALLBACK ✔  saat LLM error, tidak ada saran (profil tidak berubah)")

assert candidate["id"] not in build_skill_prompt(DEMO_STORY)
print("PRIVASI ✔  prompt tidak memuat id pengguna")

DITOLAK ✔  kode di luar kosakata
DITOLAK ✔  kutipan karangan (tidak ada di cerita)
DITOLAK ✔  tanpa kutipan
DITOLAK ✔  kutipan memuat info pribadi
DITOLAK ✔  bukan JSON
TANPA PERSETUJUAN ✔  LLM tidak dipanggil
[ekstraksi gagal -> tanpa saran] TimeoutError: API tidak merespons
FALLBACK ✔  saat LLM error, tidak ada saran (profil tidak berubah)
PRIVASI ✔  prompt tidak memuat id pengguna


## 6. Filter dan score 7 komponen
Hard filter (`published`, hijau, belum lewat deadline) lalu score 0-100 dari tujuh komponen berbobot: `skill`, `sector`, `work_style`, `location`, `experience`, `education`, `benefits`. Data missing dihitung neutral, bukan 0.

> Dataset tidak punya kolom status/deadline, jadi semua lowongan lolos; filter diuji pada 3 lowongan buatan.

In [11]:
def filter_reason(candidate, job, today, exclude_applied=EXCLUDE_APPLIED):
    # Alasan lowongan dibuang, atau None bila lolos
    if job["_status"] != "published":
        return "status bukan 'published'"
    if not job["_is_green"]:
        return "bukan lowongan hijau (is_green)"
    deadline = parse_date(job["_deadline"])
    if deadline is not None and deadline < today:
        return "deadline sudah lewat"
    if exclude_applied and job["_id"] in candidate["applied_job_ids"]:
        return "sudah dilamar"
    return None


today = date.today()

# Lowongan uji buatan sendiri: draft, bukan hijau, deadline lewat, dan satu yang normal
df_filter_test = pd.DataFrame([
    {"_id": "T1", "_title": "Draft", "_status": "draft", "_is_green": True, "_deadline": ""},
    {"_id": "T2", "_title": "Bukan hijau", "_status": "published", "_is_green": False, "_deadline": ""},
    {"_id": "T3", "_title": "Deadline lewat", "_status": "published", "_is_green": True, "_deadline": f"{today.year - 1}-01-01"},
    {"_id": "T4", "_title": "Normal", "_status": "published", "_is_green": True, "_deadline": f"{today.year + 1}-12-31"},
])
display(pd.DataFrame(
    [(r["_id"], r["_title"], filter_reason(candidate, r, today) or "pass") for _, r in df_filter_test.iterrows()],
    columns=["id", "title", "filter_result"],
))

# Filter pada dataset asli
rows = [(r["_id"], r["_title"], filter_reason(candidate, r, today) or "pass") for _, r in df_jobs.iterrows()]
df_filter = pd.DataFrame(rows, columns=["id", "title", "filter_result"])
print(f"{(df_filter['filter_result'] == 'pass').sum()} dari {len(df_jobs)} lowongan lolos filter.")

,id,title,filter_result
0,T1,Draft,status bukan 'published'
1,T2,Bukan hijau,bukan lowongan hijau (is_green)
2,T3,Deadline lewat,deadline sudah lewat
3,T4,Normal,pass


50 dari 50 lowongan lolos filter.


In [12]:
# Setiap komponen mengembalikan 0-100, atau None bila data missing (nanti diganti NEUTRAL).

def overlap(user_features, job_features):
    # % fitur yang DIMINTA lowongan dan dimiliki user (denominator = fitur lowongan)
    if not job_features or not user_features:
        return None, set()
    matched = user_features & job_features
    return 100.0 * len(matched) / len(job_features), matched


def sector_score(candidate, job):
    if not job["_sector"] or not candidate["interest_sectors"]:
        return None
    return 100.0 if norm(job["_sector"]) in candidate["interest_sectors"] else 0.0


def work_style_score(candidate, job):
    if not job["_mode"] or not candidate["work_arrangement_pref"]:
        return None
    if job["_mode"] == candidate["work_arrangement_pref"]:
        return 100.0
    # hybrid = jalan tengah antara onsite dan remote
    return 50.0 if "hybrid" in (job["_mode"], candidate["work_arrangement_pref"]) else 0.0


def user_needs(candidate):
    # Kebutuhan user -> kode benefit, dari preferences dan jalur karier
    prefs = candidate["preferences"]
    needs = {b for key, b in NEED_TO_BENEFIT.items() if prefs.get(key) in (True, "true", "True", 1)}
    if candidate["career_path"] == "returning_to_work":
        needs.add("return_to_work")
    return needs


def benefits_score(candidate, job):
    needs = user_needs(candidate)
    if not needs:
        return None, set()
    met = needs & job["_benefits"]
    return 100.0 * len(met) / len(needs), met


def location_score(candidate, job):
    if job["_mode"] == "remote":
        return 100.0
    if not job["_location"] or not candidate["province"]:
        return None
    if norm_location(job["_location"]) == norm_location(candidate["province"]):
        return 100.0
    return 70.0 if candidate["open_to_relocate"] else 0.0


def gap_score(have, need):
    return 100.0 if have >= need else 50.0 if have == need - 1 else 0.0


def experience_score(candidate, job):
    if job["_experience"] not in EXPERIENCE_RANK:
        return 100.0                       # lowongan tidak mensyaratkan
    if candidate["experience_level"] not in EXPERIENCE_RANK:
        return None
    return gap_score(EXPERIENCE_RANK[candidate["experience_level"]], EXPERIENCE_RANK[job["_experience"]])


def education_score(candidate, job):
    if job["_education"] not in EDUCATION_RANK:
        return 100.0
    if candidate["max_education"] not in EDUCATION_RANK:
        return None                        # pendidikan belum diisi = missing data, bukan tidak cocok
    return gap_score(EDUCATION_RANK[candidate["max_education"]], EDUCATION_RANK[job["_education"]])


# Uji satu pasangan: kandidat demo vs lowongan pertama
job0 = df_jobs.iloc[0]
skill_pct, skill_matched = overlap(candidate["skills"], job0["_skills"])
print("Lowongan        :", job0["_title"])
print(f"Skill           : {skill_pct}  (cocok: {sorted(skill_matched)})")
print("Sector          :", sector_score(candidate, job0))
print("Work style      :", work_style_score(candidate, job0))
print("Benefit         :", benefits_score(candidate, job0)[0], f" (kebutuhan: {sorted(user_needs(candidate))})")
print("Location        :", location_score(candidate, job0))
print("Experience      :", experience_score(candidate, job0))
print("Education       :", education_score(candidate, job0))

Lowongan        : Environment Officer
Skill           : 0.0  (cocok: [])
Sector          : 100.0
Work style      : 100.0
Benefit         : 0.0  (kebutuhan: ['return_to_work'])
Location        : 0.0
Experience      : 100.0
Education       : 100.0


## 7. Rank, penjelasan, dan dampak skill
`recommend_jobs()` = filter → score → urutkan, lengkap dengan alasan dan skill gap. Skor adalah kesesuaian, **bukan** peluang diterima.

In [13]:
def explain(job, c, matched, benefits_met):
    # Alasan singkat. Hanya menyebut benefit yang DISEDIAKAN lowongan, tidak pernah kondisi pribadi user
    reasons = []
    if matched:
        reasons.append(f"{len(matched)} dari {len(job['_skills'])} skill yang diminta sudah Anda miliki")
    if c["sector"] == 100:
        reasons.append("sektor sesuai minat Anda")
    if c["work_style"] == 100 and job["_mode"]:
        reasons.append(f"pola kerja {job['_mode']} sesuai preferensi Anda")
    if benefits_met:
        reasons.append("menyediakan " + ", ".join(BENEFIT_LABEL[b] for b in sorted(benefits_met)))
    if c["location"] == 100:
        reasons.append("bisa dikerjakan remote" if job["_mode"] == "remote" else "lokasi sesuai domisili Anda")
    if c["experience"] == 100 and job["_experience"]:
        reasons.append("pengalaman Anda memenuhi syarat")
    if c["education"] == 100 and job["_education"]:
        reasons.append("pendidikan Anda memenuhi syarat")
    return reasons or ["skor kesesuaian total relatif tinggi dibanding lowongan lain"]


def score_job(candidate, job, weights=WEIGHTS):
    skill, matched = overlap(candidate["skills"], job["_skills"])
    ben, ben_met = benefits_score(candidate, job)
    parts = {
        "skill": skill, "sector": sector_score(candidate, job), "work_style": work_style_score(candidate, job),
        "benefits": ben, "location": location_score(candidate, job),
        "experience": experience_score(candidate, job), "education": education_score(candidate, job),
    }
    total = sum(weights[k] * (NEUTRAL if v is None else v) for k, v in parts.items())
    label = lambda code: job["_skill_labels"].get(code, code.replace("_", " "))
    components = {k: None if v is None else round(v, 1) for k, v in parts.items()}
    return {
        "job_id": job["_id"], "title": job["_title"], "company": job["_company"],
        "location": job["_location"], "work_mode": job["_mode"],
        "score": round(total, 1), "components": components,
        "known_dimensions": sum(v is not None for v in parts.values()),
        "reasons": explain(job, parts, matched, ben_met),
        "matched_skills": sorted(label(s) for s in matched),
        "skill_gap": sorted(label(s) for s in job["_skills"] - candidate["skills"]),
        "already_applied": job["_id"] in candidate["applied_job_ids"],
    }


def recommend_jobs(candidate, jobs=None, top_n=TOP_N, weights=WEIGHTS, exclude_applied=EXCLUDE_APPLIED, today=None):
    # FILTER -> SCORE -> RANK. Mengembalikan (recs, skipped)
    jobs = df_jobs if jobs is None else jobs
    today = today or date.today()

    skipped, scored = [], []
    for _, job in jobs.iterrows():
        reason = filter_reason(candidate, job, today, exclude_applied)
        if reason:
            skipped.append((job["_title"], reason))
        else:
            scored.append(score_job(candidate, job, weights))

    recs = pd.DataFrame(scored)
    if len(recs):
        recs = recs.sort_values("score", ascending=False, kind="stable").head(top_n).reset_index(drop=True)
        recs.index = recs.index + 1   # peringkat mulai dari 1
    return recs, skipped


def to_frame(recs):
    # Tabel ringkas untuk ditampilkan
    return pd.DataFrame([
        {"#": i, "title": r["title"], "company": r["company"] or "-", "lokasi": r["location"] or "-",
         "score": r["score"], **r["components"],
         "known_dimensions": r["known_dimensions"], "already_applied": r["already_applied"]}
        for i, r in recs.iterrows()
    ]).set_index("#")


def print_details(recs, n=5):
    for i, r in recs.head(n).iterrows():
        print(f"\n{i}. {r['title']} - {r['company'] or '(perusahaan tidak dicantumkan)'} [{r['score']}/100]"
              + ("  (sudah dilamar)" if r["already_applied"] else ""))
        print("   Reasons  :", "; ".join(r["reasons"]) + ".")
        if r["skill_gap"]:
            print("   Skill gap:", ", ".join(r["skill_gap"]))


recs, skipped = recommend_jobs(candidate)
display(to_frame(recs))
print(f"Ditolak filter: {len(skipped)} lowongan")
print_details(recs, n=5)

,title,company,lokasi,score,skill,sector,work_style,benefits,location,experience,education,known_dimensions,already_applied
#,,,,,,,,,,,,,
1,Environment Engineer,Petrosea,-,66.7,26.7,100.0,100.0,0.0,NaN,100.0,100.0,6,True
2,HSE Officer,-,West Jakarta,56.9,7.7,100.0,100.0,0.0,0.0,100.0,100.0,7,False
3,Oil Refinery,Human Protalent Indonesia,-,56.8,27.3,NaN,100.0,0.0,NaN,100.0,100.0,5,False
4,Environment Officer,Anzawara Satria,Tanah Bumbu,55.0,0.0,100.0,100.0,0.0,0.0,100.0,100.0,7,False
5,Supervisor Infrastructure,Tatamulia Nusantara Indah,-,52.5,NaN,0.0,100.0,0.0,NaN,100.0,100.0,5,False
6,Project Compliance Officer,Merdeka Copper Gold,South Jakarta,50.0,20.0,0.0,100.0,0.0,100.0,100.0,100.0,7,False
7,Ahli Resolusi konflik/ Sosio - Ekonomi,Permata Karya Jasa,-,50.0,0.0,NaN,100.0,0.0,NaN,100.0,100.0,5,False
8,QHSE,-,South Jakarta,50.0,10.0,NaN,100.0,0.0,100.0,50.0,100.0,6,False
9,"Business Development Manager (Oil & Gas, PS & ...",SGS,South Jakarta,48.6,14.3,0.0,100.0,0.0,100.0,100.0,100.0,7,False


Ditolak filter: 0 lowongan

1. Environment Engineer - Petrosea [66.7/100]  (sudah dilamar)
   Reasons  : 4 dari 15 skill yang diminta sudah Anda miliki; sektor sesuai minat Anda; pola kerja onsite sesuai preferensi Anda; pengalaman Anda memenuhi syarat; pendidikan Anda memenuhi syarat.
   Skill gap: AMDAL / UKL-UPL / RKL-RPL, Corrective & preventive actions (CAPA), Environmental management, Environmental monitoring, Environmental regulations & compliance, Environmental reporting & documentation, Heavy equipment maintenance & repair, PROPER, Safety inspection & audit, Stakeholder / government relations, Waste management (B3/non-B3)

2. HSE Officer - (perusahaan tidak dicantumkan) [56.9/100]
   Reasons  : 1 dari 13 skill yang diminta sudah Anda miliki; sektor sesuai minat Anda; pola kerja onsite sesuai preferensi Anda; pendidikan Anda memenuhi syarat.
   Skill gap: Attention to detail, Community relations / CSR, Corrective & preventive actions (CAPA), Emergency response, HSE / HSEQ / QHS

**Interpretasi:**
- Peringkat 1 (`Environment Engineer`, 66,7) **sudah dilamar**. Di UI ia akan berlabel *Sudah Dilamar*; ubah `EXCLUDE_APPLIED = True` bila ingin menyembunyikannya.
- Peringkat 2 dan 3 hanya berbeda 0,1 poin (56,9 dan 56,8). Selisih sekecil ini tidak boleh dibaca sebagai "pasti lebih baik".
- Skor tertinggi hanya 66,7. Penyebabnya, profil awal hanya punya 4 skill sehingga komponen `skill` rendah, dan komponen `benefits` bernilai 0 untuk **semua** lowongan: profil `returning_to_work` mencari program return-to-work, sedangkan dataset tidak mencantumkannya. Skor dibaca sebagai kesesuaian *relatif* antar lowongan, bukan nilai absolut.
- `Environment Engineer` punya `known_dimensions = 6` karena lokasi lowongan kosong. Komponen `location` missing (`NaN`) dan dihitung 50, bukan 0.
- `Supervisor Infrastructure` (peringkat 5, skor 52,5) skill-nya `[Perlu review manual]`, jadi komponen `skill` missing dan dihitung 50. Akibatnya lowongan ini terangkat walau sektornya tidak cocok. Ini efek kebijakan *missing = neutral*: lowongan dengan data tidak lengkap bisa tampil tinggi, dan datanya perlu dilengkapi.

**Dampak skill hasil ekstraksi** (sebelum vs sesudah dikonfirmasi):

In [14]:
recs_before, _ = recommend_jobs(candidate, top_n=1000)
recs_after, _ = recommend_jobs(candidate_enriched, top_n=1000)

cmp = pd.DataFrame({
    "title": recs_before.set_index("job_id")["title"],
    "skor_sebelum": recs_before.set_index("job_id")["score"],
    "skor_sesudah": recs_after.set_index("job_id")["score"],
    "peringkat_sebelum": pd.Series(recs_before.index, index=recs_before["job_id"]),
    "peringkat_sesudah": pd.Series(recs_after.index, index=recs_after["job_id"]),
})
cmp["selisih_skor"] = (cmp["skor_sesudah"] - cmp["skor_sebelum"]).round(1)

assert (cmp["selisih_skor"] >= 0).all(), "Menambah skill tidak boleh menurunkan skor"
display(cmp.sort_values("peringkat_sesudah").head(10).set_index("peringkat_sesudah"))

,title,skor_sebelum,skor_sesudah,peringkat_sebelum,selisih_skor
peringkat_sesudah,,,,,
1,Environment Engineer,66.7,75.0,1,8.3
2,Environment Officer,55.0,64.4,4,9.4
3,HSE Officer,56.9,62.7,2,5.8
4,Oil Refinery,56.8,59.1,3,2.3
5,Project Compliance Officer,50.0,57.5,6,7.5
6,Environment Superitendent,45.0,57.5,14,12.5
7,QHSE,50.0,56.2,8,6.2
8,Ahli Resolusi konflik/ Sosio - Ekonomi,50.0,54.2,7,4.2
9,Supervisor Infrastructure,52.5,52.5,5,0.0


**Interpretasi:** setelah tujuh skill hasil ekstraksi dikonfirmasi, `Environment Engineer` naik dari 66,7 ke 75,0, `Environment Officer` naik dari peringkat 4 ke 2 (55,0 → 64,4), dan `Environment Superintendent` naik dari peringkat 14 ke 6. Lowongan yang skill-nya missing (`Supervisor Infrastructure`) tidak berubah dan turun dari peringkat 5 ke 9. Tidak ada lowongan yang skornya turun.

## 8. Eksperimen weight

Weight adalah asumsi, bukan kebenaran. Bandingkan tiga skema: `default` (weight di tahap 2, tidak diubah), `skill_focus` (menekankan skill), dan `needs_focus` (menekankan benefit dan pola kerja).

In [15]:
VARIANTS = {
    "default": WEIGHTS,
    "skill_focus": {"skill": 0.60, "sector": 0.15, "work_style": 0.05, "benefits": 0.05,
                    "location": 0.05, "experience": 0.05, "education": 0.05},
    "needs_focus": {"skill": 0.20, "sector": 0.15, "work_style": 0.15, "benefits": 0.30,
                    "location": 0.10, "experience": 0.05, "education": 0.05},
}
for name, w in VARIANTS.items():
    assert abs(sum(w.values()) - 1.0) < 1e-9, f"Weights '{name}' tidak berjumlah 1.0"

table = {}
for name, w in VARIANTS.items():
    ranked, _ = recommend_jobs(candidate, top_n=1000, weights=w)
    table[name] = {r["job_id"]: (pos, r["score"]) for pos, r in ranked.iterrows()}

top_ids = list(recs_before["job_id"].head(10))   # Top-10 skema default
titles = recs_before.set_index("job_id")["title"]
display(pd.DataFrame(
    {name: [f"#{table[name][j][0]} ({table[name][j][1]})" for j in top_ids] for name in VARIANTS},
    index=[f"{titles[j]} [{j}]" for j in top_ids],
))

,default,skill_focus,needs_focus
Environment Engineer [95283657],#1 (66.7),#1 (48.5),#1 (50.3)
HSE Officer [95228875],#2 (56.9),#7 (34.6),#4 (41.5)
Oil Refinery [95527538],#3 (56.8),#4 (41.4),#2 (43.0)
Environment Officer [95465065],#4 (55.0),#11 (30.0),#5 (40.0)
Supervisor Infrastructure [95441159],#5 (52.5),#2 (47.5),#6 (40.0)
Project Compliance Officer [95573223],#6 (50.0),#9 (32.0),#7 (39.0)
Ahli Resolusi konflik/ Sosio - Ekonomi [95000794],#7 (50.0),#23 (25.0),#10 (37.5)
QHSE [95468250],#8 (50.0),#10 (31.0),#3 (42.0)
"Business Development Manager (Oil & Gas, PS & AIMS) [95582903]",#9 (48.6),#12 (28.6),#8 (37.9)
Mining Mandarin Translator [95544674],#10 (48.3),#14 (28.0),#9 (37.7)


**Interpretasi:** `Environment Engineer` tetap peringkat 1 di ketiga skema, jadi posisinya relatif stabil. Peringkat lainnya sensitif terhadap weight: `HSE Officer` turun dari #2 ke #7 pada `skill_focus` (hanya 1 dari 13 skill yang cocok), `Environment Officer` turun dari #4 ke #11, sedangkan `QHSE` naik dari #8 ke #3 pada `needs_focus`. Artinya urutan di bagian atas perlu dikalibrasi dengan data nyata (klik, simpan, lamar), bukan ditentukan sekali lalu dianggap final.

## 9. Lapisan AI: penjelasan personal
AI hanya menjelaskan Top-3 hasil aturan (score ≥ 40), tidak mengubah urutan, dan outputnya divalidasi. Bila gagal, alasan berbasis aturan dipakai.

> `stub_llm` adalah simulasi template, bukan LLM sungguhan.

In [16]:
AI_TOP_K = 3             # hanya Top-K teratas yang dijelaskan AI (hemat biaya dan lebih fokus)
AI_MIN_SCORE = 40.0      # lowongan di bawah ambang ini tidak layak dijelaskan sebagai "cocok"
USE_REAL_LLM = False     # True bila call_real_llm sudah diisi
AI_MODEL_NAME = "stub-template-v0"   # ganti dengan nama model asli saat USE_REAL_LLM = True

# SENSITIVE_TERMS (kata yang tidak boleh muncul pada output AI) didefinisikan di tahap 5

SYSTEM_PROMPT = """Kamu asisten karier Laras Karya. Tugasmu HANYA menjelaskan hasil pencocokan yang sudah dihitung sistem.
Aturan:
1. Bahas hanya lowongan pada DATA_JSON. Jangan menambah lowongan, perusahaan, gaji, atau fakta baru.
2. Jangan mengubah score atau urutan.
3. Jangan menebak kondisi pribadi pengguna (keluarga, kesehatan, usia).
4. Bahasa Indonesia yang hangat dan ringkas, gunakan sapaan 'Anda'.
5. Balas HANYA dengan JSON: {"items": [{"job_id": "...", "summary": "maks 2 kalimat", "next_step": "1 saran konkret terkait skill gap"}]}"""


def ai_shortlist(recs, k=AI_TOP_K, min_score=AI_MIN_SCORE):
    return recs[recs["score"] >= min_score].head(k)


def build_ai_payload(shortlist):
    # Hanya fakta hasil aturan. Tidak ada preferences / data sensitif / id pengguna
    return [
        {"job_id": r["job_id"], "title": r["title"], "company": r["company"], "score": r["score"],
         "matched_skills": r["matched_skills"], "skill_gap": r["skill_gap"],
         "rule_based_reasons": r["reasons"], "already_applied": r["already_applied"]}
        for _, r in shortlist.iterrows()
    ]


def build_prompt(payload):
    return SYSTEM_PROMPT + "\n\nDATA_JSON:\n" + json.dumps(payload, ensure_ascii=False)


def stub_llm(prompt):
    # SIMULASI berbasis template (bukan LLM). Membaca DATA_JSON dari prompt dan mengembalikan JSON valid
    data = json.loads(prompt.split("DATA_JSON:\n", 1)[1])
    items = []
    for d in data:
        top_reasons = "; ".join(d["rule_based_reasons"][:2])
        where = f" di {d['company']}" if d["company"] else ""
        summary = f"{d['title']}{where} cocok dengan profil Anda (skor {d['score']}/100): {top_reasons}."
        if d["already_applied"]:
            step = "Anda sudah melamar lowongan ini; pantau status lamaran Anda."
        elif d["skill_gap"]:
            step = f"Perkuat skill {d['skill_gap'][0]} untuk menutup kesenjangan sebelum melamar."
        else:
            step = "Skill Anda sudah memenuhi yang diminta; siapkan CV dan surat lamaran."
        items.append({"job_id": d["job_id"], "summary": summary, "next_step": step})
    return json.dumps({"items": items}, ensure_ascii=False)


def call_real_llm(prompt):
    # Isi dengan pemanggilan API LLM pilihan. Harus mengembalikan STRING JSON sesuai format di SYSTEM_PROMPT
    raise NotImplementedError("Sambungkan ke API LLM, lalu set USE_REAL_LLM = True")


def validate_ai_output(raw, allowed_ids):
    # Guardrail: JSON valid, job_id ada di shortlist, field wajib terisi, tanpa kata sensitif, panjang dibatasi
    try:
        data = json.loads(raw)
    except (json.JSONDecodeError, TypeError):
        return {}
    items = data.get("items", []) if isinstance(data, dict) else []
    out = {}
    for it in items:
        if not isinstance(it, dict):
            continue
        jid, s, n = str(it.get("job_id")), it.get("summary"), it.get("next_step")
        if jid not in allowed_ids or not (isinstance(s, str) and isinstance(n, str) and s.strip() and n.strip()):
            continue
        if any(t in (s + " " + n).lower() for t in SENSITIVE_TERMS):
            continue
        out[jid] = {"summary": s.strip()[:400], "next_step": n.strip()[:200]}
    return out


def ai_enrich(recs, llm=None):
    # Shortlist -> prompt -> LLM -> validasi. Gagal apa pun = {} (fallback ke alasan berbasis aturan)
    llm = llm or (call_real_llm if USE_REAL_LLM else stub_llm)
    shortlist = ai_shortlist(recs)
    if not len(shortlist):
        return {}
    try:
        raw = llm(build_prompt(build_ai_payload(shortlist)))
    except Exception as e:                       # jaringan, kuota, NotImplementedError, dst.
        print(f"[AI gagal -> fallback ke rule-based] {type(e).__name__}: {e}")
        return {}
    return validate_ai_output(raw, set(shortlist["job_id"]))


ai = ai_enrich(recs)
display(pd.DataFrame([
    {"#": i, "title": r["title"], "score": r["score"],
     "sumber_penjelasan": "AI" if r["job_id"] in ai else "rule-based",
     "ai_summary": ai.get(r["job_id"], {}).get("summary", "-"),
     "ai_next_step": ai.get(r["job_id"], {}).get("next_step", "-")}
    for i, r in recs.head(8).iterrows()
]).set_index("#"))

,title,score,sumber_penjelasan,ai_summary,ai_next_step
#,,,,,
1,Environment Engineer,66.7,AI,Environment Engineer di Petrosea cocok dengan ...,Anda sudah melamar lowongan ini; pantau status...
2,HSE Officer,56.9,AI,HSE Officer cocok dengan profil Anda (skor 56....,Perkuat skill Attention to detail untuk menutu...
3,Oil Refinery,56.8,AI,Oil Refinery di Human Protalent Indonesia coco...,Perkuat skill Analytical thinking untuk menutu...
4,Environment Officer,55.0,rule-based,-,-
5,Supervisor Infrastructure,52.5,rule-based,-,-
6,Project Compliance Officer,50.0,rule-based,-,-
7,Ahli Resolusi konflik/ Sosio - Ekonomi,50.0,rule-based,-,-
8,QHSE,50.0,rule-based,-,-


In [17]:
allowed = set(ai_shortlist(recs)["job_id"])
first_id = sorted(allowed)[0]

bad_cases = {
    "lowongan karangan (J99 tidak ada di shortlist)":
        json.dumps({"items": [{"job_id": "J99", "summary": "Gaji 50 juta!", "next_step": "Lamar sekarang"}]}),
    "bukan JSON": "Tentu! Berikut rekomendasi saya...",
    "menebak kondisi pribadi":
        json.dumps({"items": [{"job_id": first_id, "summary": "Cocok untuk ibu yang sedang menyusui.", "next_step": "Lamar"}]}),
}
for name, raw in bad_cases.items():
    result = validate_ai_output(raw, allowed)
    assert result == {}, f"Guardrail gagal untuk: {name}"
    print(f"DITOLAK ✔  {name}")

mixed = json.dumps({"items": [
    {"job_id": "J99", "summary": "karangan", "next_step": "x"},
    {"job_id": first_id, "summary": "Penjelasan valid.", "next_step": "Langkah valid."},
]})
assert list(validate_ai_output(mixed, allowed)) == [first_id]
print("CAMPURAN ✔  hanya item valid yang lolos:", list(validate_ai_output(mixed, allowed)))

assert ai_enrich(recs, llm=broken_llm) == {}
print("FALLBACK ✔  saat AI error, hasil kosong -> tampilan memakai alasan berbasis aturan")

DITOLAK ✔  lowongan karangan (J99 tidak ada di shortlist)
DITOLAK ✔  bukan JSON
DITOLAK ✔  menebak kondisi pribadi
CAMPURAN ✔  hanya item valid yang lolos: ['95228875']
[AI gagal -> fallback ke rule-based] TimeoutError: API tidak merespons
FALLBACK ✔  saat AI error, hasil kosong -> tampilan memakai alasan berbasis aturan


## 10. Uji otomatis, simpan hasil, dan kesimpulan
15 uji otomatis, lalu hasil disimpan ke `job_recommendations_output.csv`.

In [18]:
checks = []
def check(name, condition):
    assert condition, f"GAGAL: {name}"
    checks.append({"uji": name, "hasil": "PASS"})

all_recs, _ = recommend_jobs(candidate, top_n=1000)

check("total weight = 1.0 untuk semua skema", all(abs(sum(w.values()) - 1.0) < 1e-9 for w in VARIANTS.values()))
check("lowongan draft / non-hijau / lewat deadline dibuang filter",
      [filter_reason(candidate, r, today) is not None for _, r in df_filter_test.iterrows()] == [True, True, True, False])
check("semua lowongan dataset diproses (tidak ada yang hilang)", len(all_recs) == len(df_jobs))
check("hasil terurut menurun berdasarkan score", list(all_recs["score"]) == sorted(all_recs["score"], reverse=True))
check("semua score berada di rentang 0-100", all_recs["score"].between(0, 100).all())
check("missing data = neutral, bukan nol (lokasi kosong -> location None)",
      all(r["components"]["location"] is None
          for _, r in all_recs.iterrows()
          if not df_jobs.set_index("_id").loc[r["job_id"], "_location"] and r["work_mode"] != "remote"))
check("lowongan tanpa skill -> komponen skill None (bukan 0)",
      all(r["components"]["skill"] is None
          for _, r in all_recs.iterrows()
          if not df_jobs.set_index("_id").loc[r["job_id"], "_skills"]))
check("lowongan sudah dilamar tetap tampil dengan flag", bool(all_recs.set_index("job_id").loc["95283657", "already_applied"]))
check("EXCLUDE_APPLIED = True menyembunyikan lowongan yang sudah dilamar",
      "95283657" not in set(recommend_jobs(candidate, top_n=1000, exclude_applied=True)[0]["job_id"]))
prompt_text = build_prompt(build_ai_payload(ai_shortlist(recs)))
check("prompt AI tidak memuat preferences / data sensitif / id pengguna",
      not any(s in prompt_text for s in ["needs_", "has_career_break", "preferences", candidate["id"]]))
check("AI hanya membahas lowongan di shortlist", set(ai) <= set(ai_shortlist(recs)["job_id"]))
check("AI tidak mengubah score atau urutan", list(recs["job_id"]) == list(recommend_jobs(candidate)[0]["job_id"]))
check("ekstraksi skill: semua kode dari kosakata dan kutipan ada di cerita",
      all(e["code"] in SKILL_VOCAB and e["evidence"].lower() in DEMO_STORY.lower() for e in extracted))
check("ekstraksi skill: tanpa persetujuan LLM tidak dipanggil", extract_skills(DEMO_STORY, consent=False, llm=lambda p: 1 / 0) == [])
check("skill usulan AI belum memengaruhi skor sebelum dikonfirmasi",
      list(recommend_jobs(candidate)[0]["score"]) == list(recs["score"]))
display(pd.DataFrame(checks))

,uji,hasil
0,total weight = 1.0 untuk semua skema,PASS
1,lowongan draft / non-hijau / lewat deadline di...,PASS
2,semua lowongan dataset diproses (tidak ada yan...,PASS
3,hasil terurut menurun berdasarkan score,PASS
4,semua score berada di rentang 0-100,PASS
5,"missing data = neutral, bukan nol (lokasi koso...",PASS
6,lowongan tanpa skill -> komponen skill None (b...,PASS
7,lowongan sudah dilamar tetap tampil dengan flag,PASS
8,EXCLUDE_APPLIED = True menyembunyikan lowongan...,PASS
9,prompt AI tidak memuat preferences / data sens...,PASS


In [19]:
def input_hash(candidate):
    # Hash dari input profil. Hash sama = profil tidak berubah, recompute bisa dilewati
    payload = {
        "skills": sorted(candidate["skills"]), "sectors": sorted(candidate["interest_sectors"]),
        "work": candidate["work_arrangement_pref"], "province": candidate["province"],
        "relocate": candidate["open_to_relocate"], "career": candidate["career_path"],
        "exp": candidate["experience_level"], "edu": candidate["max_education"],
        "prefs": candidate["preferences"],
    }
    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()[:16]


def to_output_rows(candidate, recs, ai=None, ai_model=None):
    # Petakan hasil ke baris siap simpan
    h, ai = input_hash(candidate), ai or {}
    return [
        {
            "user_id": candidate["id"], "job_id": r["job_id"], "title": r["title"], "company": r["company"],
            "match_score": r["score"], "known_dimensions": r["known_dimensions"],
            "breakdown": json.dumps({"components": r["components"], "matched_skills": r["matched_skills"],
                                     "skill_gap": r["skill_gap"]}, ensure_ascii=False),
            "reason": "; ".join(r["reasons"]), "already_applied": r["already_applied"], "input_hash": h,
            "ai_explanation": ai.get(r["job_id"], {}).get("summary"),
            "ai_next_step": ai.get(r["job_id"], {}).get("next_step"),
            "ai_model": ai_model if r["job_id"] in ai else None,
        }
        for _, r in recs.iterrows()
    ]


OUTPUT_CSV = "job_recommendations_output.csv"
df_output = pd.DataFrame(to_output_rows(candidate, recs, ai=ai, ai_model=AI_MODEL_NAME))
df_output.to_csv(OUTPUT_CSV, index=False)

print(f"{len(df_output)} baris disimpan ke {OUTPUT_CSV}. Contoh baris pertama:\n")
print(json.dumps(df_output.iloc[0].to_dict(), ensure_ascii=False, indent=2, default=str))

20 baris disimpan ke job_recommendations_output.csv. Contoh baris pertama:

{
  "user_id": "demo-user",
  "job_id": "95283657",
  "title": "Environment Engineer",
  "company": "Petrosea",
  "match_score": 66.7,
  "known_dimensions": 6,
  "breakdown": "{\"components\": {\"skill\": 26.7, \"sector\": 100.0, \"work_style\": 100.0, \"benefits\": 0.0, \"location\": null, \"experience\": 100.0, \"education\": 100.0}, \"matched_skills\": [\"Communication\", \"Microsoft Office / Excel\", \"Problem solving\", \"Reporting & presentation\"], \"skill_gap\": [\"AMDAL / UKL-UPL / RKL-RPL\", \"Corrective & preventive actions (CAPA)\", \"Environmental management\", \"Environmental monitoring\", \"Environmental regulations & compliance\", \"Environmental reporting & documentation\", \"Heavy equipment maintenance & repair\", \"PROPER\", \"Safety inspection & audit\", \"Stakeholder / government relations\", \"Waste management (B3/non-B3)\"]}",
  "reason": "4 dari 15 skill yang diminta sudah Anda miliki; s

### Hasil dan interpretasi

**Hasil uji awal (`green-jobs.csv`, 50 lowongan × 1 profil demo):**
- Dataset bersih dipakai seluruhnya: 50 dari 50 lowongan diproses. Semuanya lolos filter karena dataset tidak punya kolom `status`, `is_green`, dan `deadline`; aturan filter diuji terpisah pada tiga lowongan buatan (draft, bukan hijau, lewat deadline) dan semuanya ditolak.
- Tiga lowongan teratas: `Environment Engineer` (66,7, sudah dilamar), `HSE Officer` (56,9), dan `Oil Refinery` (56,8). Selisih peringkat 2 dan 3 hanya 0,1 poin.
- Lowongan dengan data tidak lengkap (lokasi, sektor, level pengalaman, atau skill kosong) dihitung neutral, bukan 0. Efek sampingnya, lowongan dengan skill `[Perlu review manual]` bisa masuk 5 besar.
- Ekstraksi skill (tahap 5): dari satu cerita demo, 8 skill diusulkan beserta kutipan buktinya dan 7 di antaranya baru. Setelah dikonfirmasi, skor lowongan yang relevan naik (`Environment Engineer` 66,7 → 75,0). Kode di luar kosakata, kutipan karangan, kutipan berisi info pribadi, dan pemanggilan tanpa persetujuan semuanya ditolak.
- Lapisan AI hanya menyentuh Top-3 dan lolos semua uji guardrail (tahap 9 dan 10): lowongan karangan, output non-JSON, penebakan kondisi pribadi, dan error API semuanya tertangani.
- Weight **tidak diubah** dari pipeline awal, dan urutan teratas sensitif terhadap weight (tahap 8).

**Status:** pipeline berjalan penuh dari file CSV tanpa koneksi database. Penjelasan AI dan ekstraksi skill masih simulasi.

**Keterbatasan hasil ini:** hanya 50 lowongan dan satu profil demo buatan sendiri; belum ada evaluasi dengan pengguna; kualitas bahasa AI belum teruji dengan LLM nyata.

### Keterbatasan dan ide pengembangan

**Keterbatasan data dan pipeline**
- Dataset hanya **50 lowongan** dan hasilnya diuji pada satu profil demo serta tiga profil eksperimen. Belum ada evaluasi dengan pengguna nyata.
- Kolom `sector` pada dataset berisi *slug peran* (misalnya `environmental-officer`), bukan sektor industri. Komponen `sector` pada akhirnya membandingkan peran yang diminati dengan peran lowongan. Lima lowongan tidak punya nilai `sector`.
- Kolom `province` berisi **kota/kabupaten** (misalnya `South Jakarta`, `Bekasi Regency`), jadi profil diisi dengan nilai yang sama. Perbandingan lokasi mengabaikan kata `Regency/City/Kabupaten/Kota`.
- `experience_level` pada dataset hanya `Intermediate` dan `Advanced` (34 dari 50 kosong). Keduanya dipetakan ke `mid` dan `senior`; lowongan tanpa level dianggap tidak mensyaratkan pengalaman (100).
- Komponen `benefits` mencari benefit seperti jam fleksibel, return-to-work, penitipan anak, dan laktasi. **Tidak ada benefit semacam itu di dataset** (isinya Medical, Dental, Parking, dan sejenisnya), jadi bagi profil yang punya kebutuhan tersebut komponen ini bernilai 0 untuk semua lowongan, dan untuk profil tanpa kebutuhan bernilai neutral.
- Dua lowongan memiliki skill `[Perlu review manual]` sehingga komponen skill-nya missing.
- Skill pada dataset adalah teks bebas hasil ekstraksi; sinonim tidak dicocokkan.
- Weight ditentukan manual dan urutan teratas sensitif terhadap weight (tahap 8).
- Penjelasan AI dan ekstraksi skill masih simulasi (template dan kata kunci).

**Ide pengembangan**
1. Sambungkan LLM nyata pada `call_real_llm` dan `stub_skill_llm`, lalu evaluasi 20 sampel secara manual (akurat, tidak mengarang, tidak sensitif).
2. Tambahkan kolom benefit ramah pekerja perempuan ke dataset (jam fleksibel, return-to-work, laktasi) agar komponen `benefits` bermakna.
3. Samakan nilai `sector` dengan daftar sektor hijau yang baku dan petakan kota ke provinsi.
4. Kalibrasi weight dengan data klik, simpan, dan lamar.
5. **Skill gap → kursus:** cocokkan `skill_gap` dengan katalog kursus (aturan), baru AI yang merangkai sarannya.
6. Setelah pipeline stabil, sambungkan ke database (misalnya Supabase) dan simpan hasil ke tabel rekomendasi. Fungsi di notebook ini sengaja dipisah per tahap sehingga bagian pemuatan data dan penyimpanan bisa diganti tanpa mengubah logika scoring.

**Prinsip responsible AI:** rekomendasi harus membantu kandidat menemukan peluang, bukan menjadi keputusan otomatis untuk menolak atau menerima kandidat. Hindari penggunaan atribut sensitif dan uji apakah ranking merugikan kelompok tertentu.

### Tantangan diskusi
1. Apa yang berubah bila `NEUTRAL` dinaikkan ke 70 atau diturunkan ke 30? Lowongan dengan data lengkap atau yang missing data yang diuntungkan?
2. Haruskah lowongan yang sudah dilamar tampil sebagai "Paling cocok untuk Anda"? Apa dampaknya pada pengalaman pengguna?
3. Lebih dari separuh lowongan tidak mencantumkan `experience_level` dan diberi nilai 100. Apakah itu adil bagi lowongan yang mencantumkannya?
4. Bagaimana memeriksa bahwa weight tidak merugikan pengguna dengan jeda karier, misalnya lewat komponen `experience`?
5. Bila AI diizinkan *re-rank* Top-K, bagaimana mengukur bahwa hasilnya lebih baik daripada urutan aturan, dan bagaimana mencegahnya "mengarang" alasan?